In [1]:
!pip install -q roboflow tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.0/250.0 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 102.9 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
from roboflow import Roboflow

ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("2026-oss").project("ai-picture-book-page-detection")
dataset = project.version(1).download("folder")

loading Roboflow workspace...
loading Roboflow project...
Exporting format folder in progress : 94.0%
Version export complete for folder format



Extracting Dataset Version Zip to AI-Picture-Book-Page-Detection-1 in folder:: 100%|██████████| 962/962 [00:00<00:00, 10079.82it/s]


In [3]:
import os

print(dataset.location)
print(os.listdir(dataset.location))

/content/AI-Picture-Book-Page-Detection-1
['train', 'valid', 'README.roboflow.txt', 'README.dataset.txt', 'test']


In [4]:
import tensorflow as tf
from tensorflow.keras import layers, models

DATA_DIR = dataset.location

IMG_SIZE = (224, 224)
BATCH_SIZE = 16

train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_DIR, "train"),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_DIR, "valid"),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_DIR, "test"),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_ds.class_names
print(class_names)

Found 840 files belonging to 4 classes.
Found 80 files belonging to 4 classes.
Found 40 files belonging to 4 classes.
['none', 'page1', 'page2', 'page3']


In [5]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

In [6]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

model = models.Sequential([
    layers.Rescaling(1./127.5, offset=-1),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(len(class_names), activation="softmax")
])

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [7]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [8]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20
)

Epoch 1/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 45s 748ms/step - accuracy: 0.3440 - loss: 1.5180 - val_accuracy: 0.6750 - val_loss: 0.9790
Epoch 2/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 38s 687ms/step - accuracy: 0.6214 - loss: 0.9371 - val_accuracy: 0.8750 - val_loss: 0.5919
Epoch 3/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 43s 717ms/step - accuracy: 0.8107 - loss: 0.6167 - val_accuracy: 0.9500 - val_loss: 0.3915
Epoch 4/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 39s 668ms/step - accuracy: 0.8857 - loss: 0.4396 - val_accuracy: 0.9750 - val_loss: 0.2698
Epoch 5/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 42s 693ms/step - accuracy: 0.9405 - loss: 0.3227 - val_accuracy: 0.9875 - val_loss: 0.1969
Epoch 6/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 35s 657ms/step - accuracy: 0.9512 - loss: 0.2558 - val_accuracy: 1.0000 - val_loss: 0.1551
Epoch 7/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 36s 672ms/step - accuracy: 0.9714 - loss: 0.1984 - val_accuracy: 1.0000 - val_loss: 0.1181
Epoch 8/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 43s 704ms/step - accuracy: 0.9726 - loss: 0.1678 - val_accu

In [9]:
test_loss, test_acc = model.evaluate(test_ds)
print("Test Accuracy:", test_acc)

3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 420ms/step - accuracy: 1.0000 - loss: 0.0303
Test Accuracy: 1.0


In [10]:
model.save("/content/page_classifier_mobilenetv2.keras")